# Mosaic and grain imaging of a map of Laue patterns (no indexation)

The intensity of chosen regions (ROIs) of the detector is read in every image of a 2D map (`amesh`, `fscan2d` ...):

| step | what | result |
|---|---|---|
| 1 | choose a dataset (folders of `RAW_DATA`, filter on its name) and one of its scans | |
| 2 | one Laue pattern of the map: peak search, positions picked with the mouse, regular grid of ROIs | ROI centers |
| 3 | **mosaic**: the pixels of each picked ROI in every image, arranged as the map | one big image per ROI |
| 4 | **grain imaging**: map of the intensity of each Laue spot (a grain lights its spots where it is) | maps, overview, browser |

**How to use it**
- edit the **SETTINGS** cell, then run the cells one after the other (`Shift + Enter`)
- results are written in `PROCESSED_DATA` (mirror folder of the images, `OUTPUT`) as `roicounters_<hash>.h5`; computed again only if the parameters change (`RECOMPUTE = True` to force it). Images are only read
- interactive figures (zoom, mouse hovering, clicks to pick positions) need `ipympl` (`%matplotlib widget`)
- computation in this notebook (`RESOURCE = 'kernel'`), or as a job: this machine in background (`'local'`) or SLURM (`'magnifix'`, `'magnifix3'`: 3 nodes, `'nice-hpc7'` ...): the notebook waits for its end

Code: `LaueTools/grainimaging.py` (browser, picker, plots), `LaueTools/scripts/workflows.py` (ROI counters), `LaueTools/scripts/roicountersplot.py` (maps, browser). Indexation of the map: `../quickstart/laue_maps_quickstart.ipynb`.

**This notebook is part of the LaueTools package** (https://github.com/BM32ESRF/lauetools)

# SETTINGS

In [ ]:
# 1. experiment: datasets of <EXP_PARENT>/<EXP_ID>/bm32/<date>/RAW_DATA/<sample>/<dataset>
EXP_ID = 'a321217'             # experiment (proposal)
EXP_PARENT = '/data/visitor'   # parent folder of the experiments
EXP_DATES = None               # None: all the dates (sessions), or e.g. ['20260707']
RAW_DATA_FOLDER = None         # or a RAW_DATA folder (then EXP_ID, EXP_PARENT, EXP_DATES are not used)
DATASET_FILTER = 'map'         # filter of the menu: words of the dataset name ('-word': without it), '': all
# or without the menus: .h5 file of the dataset and index of the scan (table of the scans of STEP 1)
H5FILE = None                  # e.g. '/data/visitor/a321217/bm32/20260707/RAW_DATA/LaB6_2/LaB6_2_map2D/LaB6_2_map2D.h5'
SCAN_INDEX = None
CCDLABEL = None                # None: detector found from the image files

# 2. one Laue pattern of the map
IMAGE_INDEX = None             # None: middle of the map
PEAKSEARCH = {'IntensityThreshold': 200}   # changes of the peak search parameters (as in peaksearch.yaml)
BOXSIZE = (8, 8)               # half sizes (X, Y) of the ROIs (pixel): ROI of (2*8+1) x (2*8+1) pixels

# ROIs of the mosaic (3) and of the grain imaging (4): 'picked' (positions picked in the browser of STEP 2), 'peaks'
# (most intense peaks of the last peak search of the browser), 'grid' (GRID_STEP), or several: 'picked+peaks' ...
# 3. mosaic
MOSAIC_FROM = 'picked'
MOSAIC_ROIS = []               # [[X, Y], ...] (pixel) added to the picked positions, e.g. [[1108, 1299], [1005, 997]]
MOSAIC_NB_PEAKS = 10           # 'peaks': nb of most intense peaks (mosaic data: nb images x nb ROIs x ROI pixels)

# 4. grain imaging
GRAIN_FROM = 'peaks'
NB_PEAK_ROIS = 100             # 'peaks': nb of most intense peaks (None: all)
GRID_STEP = None               # pixel (or (stepX, stepY)), e.g. 100: regular grid of ROIs for 'grid'; None: no grid
GRID_AROUND_PICKED = False     # True: grid only in the rectangle around the picked positions (+- GRID_MARGIN)
GRID_MARGIN = 150
COUNTER = 'max'                # counter of the maps: 'max', 'mean', 'sum', 'ptp' ...

# computation
RESOURCE = 'kernel'            # 'kernel' (this notebook), 'local', 'magnifix', 'magnifix3', 'nice-hpc7' ...
NB_CPUS = None                 # None: all cpus (of a node for SLURM)
OUTPUT = 'mirror'              # results folder: 'mirror' (PROCESSED_DATA), 'image' (folder of images) or a path
RECOMPUTE = False              # True: compute again even if results of the same parameters exist

# IMPORTS

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import matplotlib.pyplot as plt
try:
    import ipympl  # interactive plots (zoom, hover, clicks) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline

import LaueTools.grainimaging as GI
import LaueTools.scripts.roicountersplot as RCP
import LaueTools.scripts.workflowlauncher as WL
print('computing resources:', ['kernel'] + list(WL.COMPUTING_RESOURCES))

# STEP 1: choose a dataset and a scan

All the datasets of the experiment are listed from the folders of `RAW_DATA` (no `.h5` file read: fast, also for thousands of datasets). Type words of the dataset name in `filter` (`map2D -align`: with `map2D`, without `align`), select a dataset: its scans are read from its own `.h5` file (`images`: image files found / expected). Select a scan, then run the next cell.

With `H5FILE` and `SCAN_INDEX` set in SETTINGS, the menus are not needed.

In [ ]:
if H5FILE is None:
    datasets = GI.find_datasets(EXP_ID, parent=EXP_PARENT, dates=EXP_DATES, raw_data_folder=RAW_DATA_FOLDER)
    selector = GI.DatasetSelector(datasets, CCDLABEL, contains=DATASET_FILTER)

In [ ]:
params = GI.scan_params(H5FILE, SCAN_INDEX, CCDLABEL) if H5FILE is not None else selector.scan_params()
GI.print_scan(params)
print(f"\nH5FILE = '{params['h5file']}'\nSCAN_INDEX = {params['scan_index']}")

# STEP 2: browser of the Laue patterns: peaks, picked positions, grid

- image index: slider (or box); the zoom is kept when the image changes. Contrast: one slider (vmin, vmax)
- **peak search** button: peak search of the current image (`PEAKSEARCH`), green `+` (checkbox: peak search of each new image). No peak search before the button is pressed. The peaks of the last peak search are used in STEP 4. `more local maxima than NumberMaxofFits` (red): no peak fitted, increase `IntensityThreshold` or `NumberMaxofFits` in `PEAKSEARCH`
- blue dots: grid of ROIs (`GRID_STEP`), red squares: ROIs of the mosaic
- **click**: picks a position (snapped on the nearest peak, or on the brightest pixel of the ROI around the click). Not while zoom or pan is active in the toolbar
- **right click** (or key `d`): removes the nearest picked position

Mouse hovering: X, Y (pixel) and intensity. The picked positions are `browser.picked` (printed by the next cell: copy them into `MOSAIC_ROIS` to keep them).

In [ ]:
plt.close('all')   # figures of previous runs of the cells
grid = (GI.grid_roicenters(params['CCDLabel'], GRID_STEP, BOXSIZE)
        if GRID_STEP and not GRID_AROUND_PICKED else None)
browser = GI.LaueImageBrowser(params, IMAGE_INDEX, peaksearch=PEAKSEARCH, boxsize=BOXSIZE, grid=grid,
                              picked=MOSAIC_ROIS)

In [ ]:
print(f'MOSAIC_ROIS = {browser.picked}')

# STEP 3: mosaic

ROIs of `MOSAIC_FROM`: the positions picked in the browser (`'picked'`), the `MOSAIC_NB_PEAKS` most intense peaks of its last peak search (`'peaks'`), the grid (`'grid'`) or several (`'picked+peaks'`). Their pixels in every image, arranged as the map (fast axis horizontal, slow axis vertical, origin at the bottom; each map point = a ROI of (2 `BOXSIZE` + 1) pixels). Mouse hovering: image index, map position, intensity.

In [ ]:
mosaic_rois = browser.roicenters(MOSAIC_FROM, MOSAIC_NB_PEAKS)
if mosaic_rois:
    res_mosaic = GI.run(params, mosaic_rois, counters=['mosaic', 'max'], boxsize=BOXSIZE, output=OUTPUT,
                        resource=RESOURCE, nb_cpus=NB_CPUS, recompute=RECOMPUTE)

In [ ]:
plt.close('all')
if mosaic_rois:
    fig, axes = GI.plot_mosaics(res_mosaic, roiindices=range(min(12, len(mosaic_rois))))   # 12 first ROIs
    plt.show()

# STEP 4: grain imaging

ROIs of `GRAIN_FROM`: the `NB_PEAK_ROIS` most intense peaks of the last peak search of STEP 2 (`'peaks'`), the picked positions (`'picked'`), a regular grid (`'grid'`, `GRID_STEP`, only around the picked positions with `GRID_AROUND_PICKED`) or several (`'peaks+grid'`). A grain lights its Laue spots only where it is: the map of the intensity of a spot is an image of the grain (or of the grains) giving this spot.

**Overview**: the intensity of each ROI is scaled to [0, 1] (background to brightest points), then at each map point: nb of ROIs lit, sum of the scaled intensities, brightest ROI (same color: same spots, i.e. likely the same grain).

In [ ]:
grid = (GI.grid_roicenters(params['CCDLabel'], GRID_STEP, BOXSIZE, margin=GRID_MARGIN,
                           around=browser.picked if GRID_AROUND_PICKED else None) if GRID_STEP else None)
grain_rois = browser.roicenters(GRAIN_FROM, NB_PEAK_ROIS, grid=grid)
if grain_rois:
    res_grains = GI.run(params, grain_rois, counters=[COUNTER], boxsize=BOXSIZE, output=OUTPUT, resource=RESOURCE,
                        nb_cpus=NB_CPUS, recompute=RECOMPUTE)

In [ ]:
plt.close('all')
fig, axes = GI.plot_grain_overview(res_grains, COUNTER, threshold=0.5)
plt.show()

## maps of the ROIs

Each map has its own contrast (percentiles of its values), the zoom is shared by all the maps. Mouse hovering: image index and value.

In [ ]:
plt.close('all')
ROI_INDICES = range(min(20, len(grain_rois)))   # e.g. [0, 3, 7] or range(0, len(grain_rois), 10)
fig, axes = RCP.plot_counter_maps(res_grains, COUNTER, roiindices=ROI_INDICES, nbcols=5, percentiles=(1, 99))

## browser of the maps

- left: map of the current ROI (slider or play), counter, contrast, log10. Click: image index and file
- right: ROI centers on the detector. Click close to a ROI center: map of this ROI

In [ ]:
browser = RCP.CounterMapBrowser(res_grains, counter=COUNTER, iroi=0)
browser.show()

## Export of this notebook (HTML, PDF)

Save the notebook first (Ctrl+S): the export reads the saved `.ipynb` file, with its outputs. Files `<notebook name>.html` / `.pdf`. Same as in a terminal: `lauetools-export-notebook <notebook>.ipynb`

In [ ]:
from LaueTools.notebook_export import export_this_notebook

EXPORT_TO = ['html', 'pdf']
EXPORT_FOLDER = None    # None: folder of the notebook
export_this_notebook(to=EXPORT_TO, outdir=EXPORT_FOLDER);